<div dir="rtl">

# 📊 05 - DocArray & Scikit-Learn Vector Stores

## لماذا نحتاج مستودعات خفيفة مثل DocArray و Scikit-Learn؟
1. **بدون قواعد بيانات أو خوادم خارجية**: تعمل مباشرة عبر مكتبات Python القياسية مثل `scikit-learn` و `docarray`.
2. **دعم خوارزميات المسافات المتنوعة**: Cosine, Euclidean (L2), Manhattan (L1), Minkowski.
3. **حفظ وتصدير بصيغ متعددة**: إمكانية حفظ الفهرس كملف `json` أو `parquet` أو `npy`.

---

### 💡 المكونات:
- **`SKLearnVectorStore`**: مستودع يعتمد على خوارزميات `NearestNeighbors` في Scikit-Learn مع دعم حفظ الفهرس في ملفات محلية.
- **`DocArrayInMemorySearch`**: مستودع متجهات فائق الخفة مدمج بذاكرة DocArray.

</div>


<div dir="rtl">

### 1️⃣ تجهيز البيئة ونموذج التضمين

</div>


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import SKLearnVectorStore, DocArrayInMemorySearch
from langchain_core.documents import Document

load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 2️⃣ إنشاء `SKLearnVectorStore` مع تحديد مقياس المسافة (Distance Metric)
يمكن تحديد نوع مقياس التشابه عبر معامل `algorithm="kd_tree"` أو `algorithm="brute"` مع `metric="cosine"` أو `metric="euclidean"`.

</div>


In [ ]:
documents = [
    Document(page_content="تعلم بايثون يفتح آفاقاً واسعة في تحليل البيانات والذكاء الاصطناعي.", metadata={"subject": "python"}),
    Document(page_content="مكتبة Scikit-Learn توفر خوارزميات كلاسيكية لتعلم الآلة والتصنيف والتجميع.", metadata={"subject": "ml"}),
    Document(page_content="محركات البحث المتجهية تساعد في بناء أنظمة التوصية وأنظمة الـ RAG.", metadata={"subject": "search"}),
    Document(page_content="قواعد البيانات العلاقية مثل PostgreSQL ممتازة لإدارة البيانات الجدولية المنظمة.", metadata={"subject": "database"})
]

# إنشاء مستودع Scikit-Learn بمقياس Cosine Similarity
sklearn_store = SKLearnVectorStore.from_documents(
    documents=documents,
    embedding=embeddings,
    algorithm="brute",
    metric="cosine"
)

print("✅ تم إنشاء SKLearnVectorStore بنجاح!")


<div dir="rtl">

### 3️⃣ البحث الدلالي وحفظ الفهرس بصيغة JSON / Parquet

</div>


In [ ]:
query = "كيف نطبق خوارزميات تعلم الآلة الكلاسيكية؟"
results = sklearn_store.similarity_search(query, k=2)

print(f"🔍 نتائج البحث للاستعلام '{query}':\n")
for doc in results:
    print(f"• {doc.page_content} (Subject: {doc.metadata['subject']})")

# حفظ الفهرس إلى القرص
sklearn_save_path = "../../data/sklearn_index.json"
sklearn_store.persist(sklearn_save_path)
print(f"\n💾 تم حفظ فهرس SKLearn بنجاح في: {os.path.abspath(sklearn_save_path)}")


<div dir="rtl">

### 4️⃣ إعادة تحميل فهرس SKLearn من القرص

</div>


In [ ]:
# تحميل الفهرس المحفوظ مسبقاً
loaded_sklearn_store = SKLearnVectorStore(
    embedding=embeddings,
    persist_path=sklearn_save_path,
    serializer="json"
)

loaded_results = loaded_sklearn_store.similarity_search("قواعد البيانات وأنظمة الجداول", k=1)
print("🎯 النتيجة من الفهرس المحمّل:")
print(f"• {loaded_results[0].page_content}")


<div dir="rtl">

### 5️⃣ تجربة `DocArrayInMemorySearch`
مستودع سريع جداً في الذاكرة للبحث الفوري دون أي إعدادات إضافية.

</div>


In [ ]:
docarray_store = DocArrayInMemorySearch.from_documents(
    documents=documents,
    embedding=embeddings
)

docarray_results = docarray_store.similarity_search_with_score("الذكاء الاصطناعي وبايثون", k=2)

print("🎯 نتائج DocArrayInMemorySearch:")
for doc, score in docarray_results:
    print(f"الدرجة: {score:.4f} | المحتوى: {doc.page_content}")
